In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import numpy as np


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving data.csv to data.csv


In [ ]:
# Load essential song-level features ONLY
data = pd.read_csv(
    "data.csv",
    usecols=[
        'id','name','artists','year',
        'acousticness','danceability','energy',
        'instrumentalness','liveness','speechiness',
        'tempo','valence'
    ]
)

print("Dataset Shape:", data.shape)
data.head()


Dataset Shape: (170653, 12)


,valence,year,acousticness,artists,danceability,energy,id,instrumentalness,liveness,name,speechiness,tempo
0,0.0594,1921,0.982,"['Sergei Rachmaninoff', 'James Levine', 'Berli...",0.279,0.211,4BJqT0PrAfrxzMOxytFOIz,0.878000,0.665,"Piano Concerto No. 3 in D Minor, Op. 30: III. ...",0.0366,80.954
1,0.9630,1921,0.732,['Dennis Day'],0.819,0.341,7xPhfUan2yNtyFG0cUWkt8,0.000000,0.160,Clancy Lowered the Boom,0.4150,60.936
2,0.0394,1921,0.961,['KHP Kridhamardawa Karaton Ngayogyakarta Hadi...,0.328,0.166,1o6I8BglA6ylDMrIELygv1,0.913000,0.101,Gati Bali,0.0339,110.339
3,0.1650,1921,0.967,['Frank Parker'],0.275,0.309,3ftBPsC5vPBKxYSee08FDH,0.000028,0.381,Danny Boy,0.0354,100.109
4,0.2530,1921,0.957,['Phil Regan'],0.418,0.193,4d6HGyGT8e121BsdKmw9v6,0.000002,0.229,When Irish Eyes Are Smiling,0.0380,101.665


In [ ]:
data = data.dropna()
data = data.reset_index(drop=True)
print("After removing missing rows:", data.shape)


After removing missing rows: (170653, 12)


In [ ]:
features = [
    'acousticness', 'danceability', 'energy',
    'instrumentalness', 'liveness', 'speechiness',
    'tempo', 'valence'
]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(data[features])


In [ ]:
model = NearestNeighbors(
    n_neighbors=10,
    algorithm='brute',
    metric='cosine'
)

model.fit(scaled_features)


NearestNeighbors(algorithm='brute', metric='cosine', n_neighbors=10)

In [ ]:
def recommend_by_index(idx, n=5):
    distances, indices = model.kneighbors(
        [scaled_features[idx]],
        n_neighbors=n+1
    )

    print(f"\n🎧 Because you liked: {data.iloc[idx]['name']} — {data.iloc[idx]['artists']}\n")

    for dist, i in zip(distances[0][1:], indices[0][1:]):
        song = data.iloc[i]
        print(f"   ➤ {song['name']} — {song['artists']} | Year {song['year']} | Similarity: {1 - dist:.3f}")


In [ ]:
def recommend_by_name(song_name, n=5):
    matches = data[data["name"].str.lower() == song_name.lower()]

    if matches.empty:
        print("❌ Song not found.")
        return

    idx = matches.index[0]
    recommend_by_index(idx, n)


In [ ]:
recommend_by_name("Shape of You")



🎧 Because you liked: Shape of You — ['Ed Sheeran']

   ➤ Shape of You — ['Ed Sheeran'] | Year 2017 | Similarity: 1.000
   ➤ Métele Sazón — ['Luny Tunes', 'Noriega', 'Tego Calderon'] | Year 2003 | Similarity: 0.995
   ➤ El Contagio — ['Los Tigres Del Norte'] | Year 2001 | Similarity: 0.993
   ➤ El Mono De Alambre — ['El Viejo Paulino Y Su Gente'] | Year 2003 | Similarity: 0.993
   ➤ Gripa Colombiana — ['Los Tucanes De Tijuana'] | Year 2000 | Similarity: 0.992


In [ ]:
recommend_by_name("Novacane")


🎧 Because you liked: Novacane — ['Frank Ocean']

   ➤ I Miss You (Dobie Rub Part One) - Sunshine Mix — ['Björk', 'Dobie'] | Year 1996 | Similarity: 0.990
   ➤ Mean It — ['Lauv', 'LANY'] | Year 2020 | Similarity: 0.990
   ➤ We Do It For Fun Pt. 2 — ['Tha Joker'] | Year 2009 | Similarity: 0.987
   ➤ Not A Love Song — ['bülow'] | Year 2017 | Similarity: 0.987
   ➤ Locked Up — ['Akon'] | Year 2005 | Similarity: 0.987


In [ ]:
recommend_by_name("Swim Good")


🎧 Because you liked: Swim Good — ['Frank Ocean']

   ➤ worlds away — ['Lil Peep'] | Year 2020 | Similarity: 0.992
   ➤ Brenda's Got A Baby — ['2Pac'] | Year 1998 | Similarity: 0.990
   ➤ No Other Love (feat. Estelle) — ['John Legend', 'Estelle'] | Year 2008 | Similarity: 0.988
   ➤ 21 — ['DaBaby'] | Year 2018 | Similarity: 0.986
   ➤ Love's Taken Over — ['Chanté Moore'] | Year 1992 | Similarity: 0.986


In [ ]:
import joblib

joblib.dump(model, 'recommender-model.pkl')
joblib.dump(scaler, 'scaler.pkl')
data.to_csv('songs-cleaned.csv', index = False)

In [ ]:
from google.colab import files
files.download('recommender-model.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download('scaler.pkl')
files.download('songs-cleaned.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import pandas as pd

REQUIRED = [
    'id','name','artists','year',
    'acousticness','danceability','energy',
    'instrumentalness','liveness','speechiness',
    'tempo','valence'
]

df = pd.read_csv("data.csv")

missing = [c for c in REQUIRED if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[REQUIRED].copy()

df['year'] = pd.to_numeric(df['year'], errors='coerce').astype('Int64')
num_cols = [c for c in REQUIRED if c not in ('id','name','artists','year')]
for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# Drop rows with critical nulls
df = df.dropna(subset=['name','artists','year'])

# Save canonical version
df.to_csv("songs_cleaned.csv", index=False)
print("✅ Saved songs_cleaned.csv with canonical column order.")
files.download('songs_cleaned.csv')


✅ Saved songs_cleaned.csv with canonical column order.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp recommender-model.pkl /content/drive/MyDrive/
!cp scaler.pkl /content/drive/MyDrive/
!cp songs_cleaned.csv /content/drive/MyDrive/

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
